# Avaliação Final — Piloto de Cobrança Preventiva (BemLar)

Este notebook é o **esqueleto** da sua entrega. Os blocos abaixo dizem *o que* precisa ser feito; o código é seu.

Antes de escrever qualquer linha:
1. Leia `00_email_financeiro.md` inteiro.
2. Leia `dicionario.md` inteiro.
3. Abra os cinco CSVs e **olhe os valores**, não só os nomes das colunas.

> A entrega vale mais pelas decisões registradas do que pelo código. Cada bloco marcado com ❓ pede uma resposta escrita — responda em célula markdown, ali mesmo.

---
## Fase 1 — Entendimento do Negócio

❓ Em uma frase, qual é a pergunta de negócio? Qual é a unidade de análise (o que é uma linha)?

❓ O Ricardo fez 5 pedidos no e-mail. Liste os 5 e, para cada um, marque agora sua intenção: **atender**, **atender com ressalva** ou **recusar**. Você pode mudar de ideia depois — mas registre a versão inicial.

1. A pergunta do negócio é: para quais clientes eu devo ligar para que os contratos que possuo não virem prejuízo? No caso da análise, na tabela final uma linha repesentará um cliente da empresa.

2. Pedidos do Ricardo:
    - Maior acurácia possível: Vamos atender, mas tendo em vista que acurácia é uma métrica que nem sempre é confiável, pois ela causa a ilusão de alta performance;
    - Uso do campo "status_contrato": Iremos considerar;
    - Uso das informações de sexo e localização: Não iremos considerar as duas colunas, pois no contexto de treino de modelos, informações sensíveis como essas podem reforçar preconceitos e produzir modelos que carregam estigmas prejudiciais por natureza;
    - Uso do score bureau: Iremos considerar;
    - Uso da lista de motivos de atraso: Iremos considerar.

---
## Fase 2 — Entendimento dos Dados

Carregue os cinco arquivos. Lembre: separador `;`, decimal `,`, datas `dd/mm/aaaa`.

Para **cada** base, responda:
- Qual é a granularidade (o que é uma linha)?
- Quantos contratos ela cobre? Todos, ou só uma parte?
- O que significa um valor vazio nessa base? Sempre a mesma coisa?

❓ Faça um inventário de problemas de qualidade: nulos, valores impossíveis, categorias que deveriam ser a mesma. Registre em uma tabela: problema | onde | quantas linhas | é erro ou artefato | o que você fez.

In [ ]:
import pandas as pd
import numpy as np

KW = dict(sep=";", decimal=",", encoding="utf-8")

# TODO: carregar contratos, pagamentos, compras, ocorrencias_sac, score_bureau
# TODO: converter as colunas de data com pd.to_datetime(..., format="%d/%m/%Y")


In [ ]:
# TODO: inventario de qualidade - nulos, impossiveis, categorias quase-duplicadas


---
## Fase 3 — Preparação dos Dados

### 3.1 Construir o alvo

A regra está no enunciado e no dicionário. Você precisa, para cada contrato:
1. identificar a **parcela de referência**;
2. calcular a **data de referência**;
3. derivar `inadimplente_30d`.

❓ Depois de construir: qual é a prevalência de positivos? Ela é compatível com o que o Ricardo descreveu no e-mail?

❓ Compare o seu alvo com `status_contrato`. Eles concordam? Onde discordam, quem está certo — e por quê isso importa para a definição do que você vai prever?

In [ ]:
# TODO: identificar a parcela de referencia por contrato
# TODO: calcular data_referencia
# TODO: construir inadimplente_30d


### 3.2 A função de corte temporal

Está pronta. Use em **toda** base de eventos antes de agregar qualquer coisa.

In [ ]:
def filtra_por_data(df_eventos, chave, col_data, datas_ref, col_ref="data_referencia"):
    """Mantem apenas os eventos ocorridos ATE a data de referencia de cada contrato.

    df_eventos : DataFrame de eventos (pagamentos, sac, compras, consultas...)
    chave      : coluna de juncao com datas_ref (ex.: "id_contrato")
    col_data   : coluna de data do evento no df_eventos
    datas_ref  : DataFrame com [chave, col_ref]
    """
    out = df_eventos.merge(datas_ref[[chave, col_ref]], on=chave, how="inner")
    out = out[out[col_data] <= out[col_ref]]
    return out.drop(columns=[col_ref])


### 3.3 Construir as features

Uma linha por contrato. Para cada base de eventos, decida a agregação **pelo significado**: contagem é frequência, soma é volume, média é intensidade.

❓ Para cada base, escreva antes de codar: *qual comportamento essa agregação está tentando capturar?*

❓ Ausência de evento vira 0, nulo, ou mediana? A resposta é a mesma para toda coluna?

Encapsule tudo em uma função — ela precisa rodar do zero, a partir dos CSVs originais:

```python
def construir_features(caminho_dados, datas_referencia):
    ...
    return df
```

In [ ]:
# TODO: construir_features()


### 3.4 Auditoria de colunas

Antes de fechar o `features.csv`, passe **cada coluna** pelas três perguntas:

1. **Essa informação existiria no momento em que a previsão seria feita?**
2. **É uma variável protegida ou sensível?**
3. **Descreve comportamento, ou só descreve quem a pessoa é?**

❓ Monte a tabela: coluna | decisão (entra / sai) | justificativa. Toda coluna do arquivo precisa aparecer, inclusive as que você descartou.

Salve o resultado em `features.csv`.

In [ ]:
# TODO: auditoria + gravar features.csv


---
## Fase 4 — Modelagem

- Split **estratificado**.
- Um **baseline simples** (árvore de decisão) e pelo menos mais um modelo.
- Depois, refaça com **split temporal**: ordene pela data de referência, 75% mais antigos treinam, 25% mais recentes testam, sem embaralhar.

❓ O resultado do split temporal foi diferente do aleatório? O que isso diz — e o que **não** diz?

In [ ]:
# TODO: split estratificado + baseline + segundo modelo


In [ ]:
# TODO: split temporal e comparacao


---
## Fase 5 — Avaliação

A equipe faz **80 ligações**. A avaliação acontece nesse corte, não no threshold de 0,50.

1. Ordene o conjunto de teste pela probabilidade prevista (`predict_proba`).
2. Corte nos 80 primeiros.
3. Calcule **precision** e **recall** nesse recorte.
4. Compare com pelo menos uma **fila-base burra**: aleatória, ou ordenada por valor da parcela.

❓ Sua fila é melhor que a fila-base? Quanto? Se a diferença for pequena, diga isso — é um resultado legítimo.

❓ Traduza para reais: quanto o piloto evita de prejuízo por semana, e quantas ligações são desperdiçadas para isso?

❓ Você prioriza **precision** ou **recall** neste caso? Justifique pelo negócio, não pela métrica.

In [ ]:
# TODO: precision@80, recall@80, fila-base, comparacao


In [ ]:
# TODO: importancia de variaveis - e cuidado ao interpretar


### Gerar a fila

Salve `fila_80.csv` com `id_contrato` e `probabilidade`, ordenado do maior risco para o menor.

In [ ]:
# TODO: gerar fila_80.csv


---
## Fechamento

Antes de enviar, confira o checklist da seção 9 do enunciado.

Faltam ainda dois arquivos que **não** são gerados aqui:
- `decisoes.md` — o que entrou, o que saiu, cada pedido do Ricardo respondido, e o model card.
- `apresentacao.pptx` — 3 slides, para o Ricardo, não para a banca.

> Se algum resultado ficou bom demais, pergunte: *que outra coisa poderia produzir esse mesmo número?*